# Lesson 37: Transfer Learning

Training a good CNN from scratch (Lessons 33-34) took hundreds of labeled examples even for a toy task. Real target tasks are often data-starved: a handful of labeled medical scans, a new product category with 20 photos. **Transfer learning** sidesteps this by reusing a network already trained on a *different*, data-rich task, on the theory that early-layer features (edges, blobs, simple textures — Lesson 33's Sobel-like first-layer filters) are useful for almost any visual task, not just the one they were originally trained on. This lesson demonstrates it on real photos, using a CIFAR-10 subset (<a href="../references.html#krizhevsky-2009-cifar">Krizhevsky, 2009</a>).

In [ ]:
import copy
import pickle
import tarfile
import urllib.request
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Source task and target task

Set up two related but distinct tasks, both from CIFAR-10. The **source task** has plenty of data: distinguishing airplanes from automobiles, 300 training images each. The **target task** is the one we actually care about, and it's deliberately starved: distinguishing cats from trucks — two classes the source task never saw at all — with only **15 training images per class**.

In [ ]:
CIFAR_URL = 'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz'
CACHE_ROOT = Path.home() / '.cache' / 'cvintro'
CACHE_DIR = CACHE_ROOT / 'cifar-10-batches-py'

def ensure_cifar10():
    if CACHE_DIR.exists():
        return
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    archive_path = CACHE_ROOT / 'cifar-10-python.tar.gz'
    if not archive_path.exists():
        print('Downloading CIFAR-10 (~163 MB, one-time, cached under ~/.cache/cvintro)...')
        urllib.request.urlretrieve(CIFAR_URL, archive_path)
    print('Extracting...')
    with tarfile.open(archive_path) as tar:
        tar.extractall(CACHE_ROOT)

def load_cifar_batch(path):
    with open(path, 'rb') as f:
        d = pickle.load(f, encoding='bytes')
    imgs = d[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1).astype(np.float32) / 255.0
    labels = np.array(d[b'labels'], dtype=np.int64)
    return imgs, labels

ensure_cifar10()

CIFAR_LABELS = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
CID = {name: CIFAR_LABELS.index(name) for name in CIFAR_LABELS}

train_imgs, train_labels = [], []
for i in range(1, 6):
    imgs, labels = load_cifar_batch(CACHE_DIR / f'data_batch_{i}')
    train_imgs.append(imgs); train_labels.append(labels)
train_imgs, train_labels = np.concatenate(train_imgs), np.concatenate(train_labels)
test_imgs, test_labels = load_cifar_batch(CACHE_DIR / 'test_batch')

def take(imgs, labels, name, n, rng_local):
    idx = np.where(labels == CID[name])[0]
    idx = rng_local.permutation(idx)[:n]
    return imgs[idx]

data_rng = np.random.default_rng(3)

# source task: airplane vs. automobile, plenty of data
src_a = take(train_imgs, train_labels, 'airplane', 300, data_rng)
src_b = take(train_imgs, train_labels, 'automobile', 300, data_rng)
X_src = np.concatenate([src_a, src_b])
y_src = np.concatenate([np.zeros(300), np.ones(300)]).astype(np.int64)
perm = data_rng.permutation(len(X_src)); X_src, y_src = X_src[perm], y_src[perm]

# target task: cat vs. truck, neither ever seen by the source task, deliberately starved
X_tgt_train = np.concatenate([take(train_imgs, train_labels, 'cat', 15, data_rng),
                               take(train_imgs, train_labels, 'truck', 15, data_rng)])
y_tgt_train = np.array([0] * 15 + [1] * 15, dtype=np.int64)
X_tgt_test = np.concatenate([take(test_imgs, test_labels, 'cat', 150, data_rng),
                              take(test_imgs, test_labels, 'truck', 150, data_rng)])
y_tgt_test = np.array([0] * 150 + [1] * 150, dtype=np.int64)

fig, axes = plt.subplots(2, 6, figsize=(11, 4))
for ax, im in zip(axes[0], X_src[:6]):
    ax.imshow(im); ax.axis('off')
axes[0, 0].set_ylabel('source', rotation=0, labelpad=25)
for ax, im in zip(axes[1], X_tgt_train[:6]):
    ax.imshow(im); ax.axis('off')
axes[1, 0].set_ylabel('target', rotation=0, labelpad=25)
fig.suptitle('Source task (airplane vs. automobile, top) vs. target task (cat vs. truck, bottom)', y=1.02)
plt.show()

## Three strategies

1. **From scratch** — train a fresh CNN on only the 12 target images. This is the baseline: no transfer at all.
2. **Frozen backbone (feature extraction)** — pretrain a CNN backbone on the source task, then freeze its weights entirely and train only a new linear classifier on top of the features it produces for target images.
3. **Fine-tuning** — start from the same pretrained backbone, but keep updating it on the target data too, using a *much smaller* learning rate for the backbone than for the new classifier head (the backbone already encodes useful structure; large updates from just 12 examples would wreck it).

In [ ]:
class Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, 5, padding=2), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 5, padding=2), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.AdaptiveMaxPool2d(1),
        )

    def forward(self, x):
        return self.conv(x).flatten(1)

class Classifier(nn.Module):
    def __init__(self, backbone, freeze_backbone):
        super().__init__()
        self.backbone = backbone
        self.freeze_backbone = freeze_backbone
        self.fc = nn.Linear(32, 2)

    def forward(self, x):
        feat = self.backbone(x)
        if self.freeze_backbone:
            feat = feat.detach()  # no gradient flows into a frozen backbone
        return self.fc(feat)

def train_source(seed, epochs=300, lr=0.001):
    torch.manual_seed(seed)  # seed before constructing the model (Lesson 33/34)
    backbone = Backbone()
    fc = nn.Linear(32, 2)
    opt = torch.optim.Adam(list(backbone.parameters()) + list(fc.parameters()), lr=lr)
    Xt = torch.tensor(X_src).permute(0, 3, 1, 2); yt = torch.tensor(y_src)
    for _ in range(epochs):
        opt.zero_grad()
        loss = F.cross_entropy(fc(backbone(Xt)), yt)
        loss.backward()
        opt.step()
    return backbone

def train_target(backbone, freeze_backbone, seed, epochs=200, lr=0.01, backbone_lr=None):
    torch.manual_seed(seed)
    model = Classifier(backbone, freeze_backbone)
    if freeze_backbone:
        opt = torch.optim.Adam(model.fc.parameters(), lr=lr)
    elif backbone_lr is not None:
        opt = torch.optim.Adam([
            {'params': model.backbone.parameters(), 'lr': backbone_lr},
            {'params': model.fc.parameters(), 'lr': lr},
        ])
    else:
        opt = torch.optim.Adam(model.parameters(), lr=lr)
    Xtr = torch.tensor(X_tgt_train).permute(0, 3, 1, 2); ytr = torch.tensor(y_tgt_train)
    Xte = torch.tensor(X_tgt_test).permute(0, 3, 1, 2); yte = torch.tensor(y_tgt_test)
    for _ in range(epochs):
        opt.zero_grad()
        loss = F.cross_entropy(model(Xtr), ytr)
        loss.backward()
        opt.step()
    with torch.no_grad():
        return (model(Xte).argmax(1) == yte).float().mean().item()

In [ ]:
pretrained = train_source(seed=0)  # pretrain once; the point of transfer learning is reusing this

scratch_accs, frozen_accs, finetune_accs = [], [], []
for seed in range(8):
    scratch_acc = train_target(Backbone(), freeze_backbone=False, seed=seed)
    frozen_acc = train_target(copy.deepcopy(pretrained), freeze_backbone=True, seed=seed)
    finetune_acc = train_target(copy.deepcopy(pretrained), freeze_backbone=False, seed=seed, backbone_lr=0.0003)
    scratch_accs.append(scratch_acc); frozen_accs.append(frozen_acc); finetune_accs.append(finetune_acc)

print(f'{"strategy":>20} {"mean test acc":>16} {"std":>8}')
print(f'{"from scratch":>20} {np.mean(scratch_accs):>15.1%} {np.std(scratch_accs):>8.1%}')
print(f'{"frozen backbone":>20} {np.mean(frozen_accs):>15.1%} {np.std(frozen_accs):>8.1%}')
print(f'{"fine-tuned":>20} {np.mean(finetune_accs):>15.1%} {np.std(finetune_accs):>8.1%}')
print(f'\n(averaged over {len(scratch_accs)} random seeds, each reusing the same 30 target training images)')

Both transfer strategies clearly beat training from scratch, and they're also dramatically more *consistent* (lower standard deviation) — with only 30 training images, a from-scratch network's success or failure depends heavily on which 30 images it happened to get and how its random initialization lands, while a pretrained backbone starts from a much better place regardless. The source task never saw a cat or a truck, yet the low-level features it learned (edges, color blobs, textures) transferred anyway, because those features are generic to natural-image recognition, not specific to "airplane vs. automobile."

Note what fine-tuning needed to work at all: a backbone learning rate roughly 30x smaller than the classifier head's. With only 30 examples, an unrestrained backbone update would simply overfit those 30 images from scratch, discarding everything useful it learned from the 600-image source task — the same catastrophic-forgetting failure mode as Lesson 34's overfitting curve, just applied to a network that started out already knowing something.

### Exercise

1. Try `backbone_lr=0.01` (i.e. no learning-rate difference between backbone and head) in the fine-tuning call. Does fine-tuned accuracy get better or worse, and does that match the catastrophic-forgetting explanation above?
2. Try freezing *most* of the backbone but fine-tuning only its last conv layer (hint: set `requires_grad = False` on the first `Conv2d`'s parameters only). Where does that land relative to fully-frozen and fully-fine-tuned?
3. Swap the target task's classes from cat/truck to dog/automobile — a target pair that overlaps the source task's *domain* (vehicles) even more closely than cat/truck did. Does frozen-backbone accuracy improve further, and does that match the intuition that transfer works best when source and target share more underlying visual structure?